In [4]:
# pip install faiss-cpu

In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

from dotenv import load_dotenv
load_dotenv()

/var/folders/fq/kr6gv5l17pd4j572n0_n3f2c0000gn/T/ipykernel_32438/3968643131.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
/Users/rahultiwari/Documents/aziz_ai_engineering/ai-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [5]:
loader=PyPDFLoader("data_test/hr_policy_manual.pdf")
pages = loader.load()

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ".", ","],
    chunk_size=512,
    chunk_overlap=100,
)

chunks = splitter.split_documents(pages)

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)



In [6]:
ret = vectorstore.as_retriever(search_kwargs={"k": 3})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [8]:
result = ret.invoke("what is the notice period for senior managers?")
for i in result:
    print(i.page_content)
    print("--------------------------------------------------")

manager and HR Business Partner.
1.2 Notice Period
Post-confirmation, the standard notice period is 60 days for all employees up to Senior Manager
level. For Associate Director and above, the notice period is 90 days. Notice period buyout is
permitted at basic salary per day, subject to management approval.
1.3 Working Hours
Standard working hours are 9 hours per day, Monday to Friday. Core hours are 10:00 AM to 5:00
PM IST. Flexible start is permitted between 8:00 AM and 10:30 AM. Work from Home (WFH) is
--------------------------------------------------
NovaTech Solutions Pvt. Ltd.Employee HR Policy Manual — FY 2024-25 | Version 3.2 | Confidential
Section 1: Employment Terms1.1 Probation Period
All new employees at NovaTech Solutions are subject to a probation period of 6 months from their
date of joining. During the probation period, either party may terminate employment with a notice
period of 15 days. Confirmation is subject to a satisfactory performance review by the reporting
ma

In [9]:
prompt = ChatPromptTemplate.from_template(
       "Answer using context only.\nContext: {context}\nQuestion: {question}"
)


def fmt(docs):
    return "\n\n".join([doc.page_content for doc in docs])



chain = (
    {"context": ret | fmt, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

In [10]:
def ask(q):
    return chain.invoke(q), ret.invoke(q)

In [11]:
ans, src = ask("What is the notice period for senior managers?")

In [12]:
ans

'The notice period for senior managers is 60 days.'

In [14]:
for i in src:
    print(i.page_content)
    print("--------------------------------------------------")

manager and HR Business Partner.
1.2 Notice Period
Post-confirmation, the standard notice period is 60 days for all employees up to Senior Manager
level. For Associate Director and above, the notice period is 90 days. Notice period buyout is
permitted at basic salary per day, subject to management approval.
1.3 Working Hours
Standard working hours are 9 hours per day, Monday to Friday. Core hours are 10:00 AM to 5:00
PM IST. Flexible start is permitted between 8:00 AM and 10:30 AM. Work from Home (WFH) is
--------------------------------------------------
NovaTech Solutions Pvt. Ltd.Employee HR Policy Manual — FY 2024-25 | Version 3.2 | Confidential
Section 1: Employment Terms1.1 Probation Period
All new employees at NovaTech Solutions are subject to a probation period of 6 months from their
date of joining. During the probation period, either party may terminate employment with a notice
period of 15 days. Confirmation is subject to a satisfactory performance review by the reporting
ma

In [16]:
ans2, srcs2 = ask("please summarize the entire documents")
print('AI:', ans2)

for i, d in enumerate(srcs2):
    print(f"Document {i}:")
    print(d.page_content)
    print("--------------------------------------------------")

AI: The document outlines policies related to paternity leave, confidentiality, data security, and disciplinary actions. It specifies that complaints regarding bullying and discrimination must be submitted to the Internal Complaints Committee (ICC) within three months, with inquiries completed within 90 days. Employees are required to sign a Non-Disclosure Agreement (NDA) before joining, and sharing confidential data externally without legal approval can lead to termination and legal consequences. The current policy supersedes all previous HR policies, and queries can be directed to hr@novatech.in or through the HR portal.
Document 0:
to documentation.
2.4 Paternity Leave
--------------------------------------------------
Document 1:
bullying, and discrimination. Complaints must be submitted to the Internal Complaints Committee
(ICC) within 3 months of the incident. ICC completes inquiry within 90 days.
4.2 Confidentiality and Data Security
All employees must sign the NDA before joinin

In [17]:
# load all 3 documebnts into vs
files =["data_test/hr_policy_manual.pdf",
        "data_test/novacrm_product_manual.pdf",
        "data_test/annual_financial_report_2024.pdf"]


all_pages =[]
for pdf in files:
    loader=PyPDFLoader(pdf)
    pages = loader.load()
    all_pages.extend(pages)

In [18]:
splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ".", ","],
    chunk_size=512,
    chunk_overlap=100,
)

chunks = splitter.split_documents(all_pages)
vectorstore = FAISS.from_documents(chunks, embeddings)

ret_all    = vectorstore.as_retriever(search_kwargs={"k": 5})


In [19]:
chain_all = (
    {"context": ret_all | fmt, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

In [20]:
q = "What was the company's performance last year and what are employee benefits?"

ans4  = chain_all.invoke(q)



In [21]:
print('AI:', ans4)

AI: In the financial year 2022-23, NovaTech reported a total revenue of Rs. 221 Crore, a gross profit of Rs. 103 Crore, EBITDA of Rs. 42 Crore, and a net profit of Rs. 24 Crore. The employee count at the end of the year was 1,391.

Regarding employee benefits, NovaTech's compensation structure includes a salary structure where the basic salary is 40% of the CTC and fully taxable, house rent allowance (HRA) is 20% and partially exempt, special allowance is 20% and fully taxable, performance bonus is 10% and fully taxable, provident fund (employer contribution) is 5% and exempt up to limits, gratuity (employer contribution) is 3% and exempt up to Rs. 20 lakhs, and medical reimbursement is 2% and exempt up to Rs. 15,000 per annum. Additionally, the company contributes 12% of the basic salary towards the Employee Provident Fund (EPF) as per the EPF Act, 1952. Appraisals are conducted annually in April, with salary revisions communicated by March 31.


In [22]:
srcs4 = ret_all.invoke(q)

for d in srcs4:
    print("###")
    print(d.metadata['source'].split("/")[-1])
    print(d.page_content)
    print("###")



###
hr_policy_manual.pdf
Medical Reimbursement
2%
Exempt up to Rs. 15,000 p.a.
3.2 Performance Appraisal Cycle
Appraisals are conducted annually in April. Process: self-assessment, manager rating, calibration.
Salary revisions effective 1st April are communicated by March 31. Mid-year promotions may be
considered in October for exceptional performers.
3.3 Employee Provident Fund (EPF)
NovaTech contributes 12% of Basic Salary towards EPF per the EPF Act, 1952. Employee
###
###
hr_policy_manual.pdf
Section 3: Compensation and Benefits3.1 Salary Structure
The salary structure at NovaTech comprises the following components:
Component
% of CTC
Tax Treatment
Basic Salary
40%
Fully Taxable
House Rent Allowance (HRA)
20%
Partially Exempt (Sec 10(13A))
Special Allowance
20%
Fully Taxable
Performance Bonus
10%
Fully Taxable
Provident Fund (Employer)
5%
Exempt up to limits
Gratuity (Employer)
3%
Exempt up to Rs. 20 lakhs
Medical Reimbursement
2%
Exempt up to Rs. 15,000 p.a.
3.2 Performance Apprai

In [23]:
q2   = "Give me a summary of all three documents."
ans5 = chain_all.invoke(q2)
srcs5 = ret_all.invoke(q2)

In [24]:
print(ans5)

The documents cover various aspects of company policies and structures at NovaTech. 

1. **Paternity Leave**: The document outlines the paternity leave policy, categorizing it into two levels: P3 Medium, which allows 4 hours of leave for Team Leads and 24 hours for others, and P4 Low, which allows 24 hours and 72 hours respectively, with an auto-resolve feature.

2. **Compensation and Benefits**: This section details the salary structure, breaking down the components of the Cost to Company (CTC) including Basic Salary, House Rent Allowance, Special Allowance, Performance Bonus, Provident Fund, Gratuity, and Medical Reimbursement, along with their tax treatments.

3. **Performance Appraisal Cycle and Internal Complaints**: It includes guidelines for performance appraisals and addresses issues related to workplace bullying and discrimination, stating that complaints should be submitted to the Internal Complaints Committee (ICC) within three months, which will complete inquiries within 90

In [25]:
for d in srcs5:
    print("###")
    print(d.metadata['source'].split("/")[-1])
    print(d.page_content)
    print("###")

###
novacrm_product_manual.pdf
Supports GST-inclusive and GST-exclusive pricing, multi-currency quotes (INR, USD, AED, SGD),
product bundling, and volume-based discount slabs. Proposals generated as branded PDFs from
configurable templates. E-signature via Aadhaar eSign and DocuSign.
###
###
hr_policy_manual.pdf
to documentation.
2.4 Paternity Leave
###
###
novacrm_product_manual.pdf
P3 Medium
4 hours
24 hours
Team Lead
P4 Low
24 hours
72 hours
Auto-resolve
3.3 Knowledge Base
Built-in Knowledge Base supports articles, FAQs, video embeds, downloadable attachments.
Searchable via full-text and AI-semantic search. Articles can be public (customer-facing), internal
(agent-only), or partner-restricted. Auto-suggest recommends KB articles to agents, reducing
average handling time by up to 35%.
###
###
hr_policy_manual.pdf
Section 3: Compensation and Benefits3.1 Salary Structure
The salary structure at NovaTech comprises the following components:
Component
% of CTC
Tax Treatment
Basic Salary
